In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict


In [ ]:
def load_prepare_and_group_by_coordinate_chunked_grouping(args,in_dir,data_file, geo_channels, coord_csv_path, coord_axis, group_size=8, chunksize=100000):
    """
    Load PET data in chunks, preprocess and group by spatial coordinates for memory-efficient handling.

    Parameters:
        args: Command-line arguments
        geo_channels: Array mapping channels
        coord_csv_path: Path to geometry CSV
        coord_axis: Axis to group by ('x', 'y', 'z')
        group_size: How many coordinates per group
        chunksize: Rows per chunk for read_csv

    Returns:
        list: List of grouped DataFrames
    """
    def to_reco_channel_id(AbsChannelID):
        portID = AbsChannelID // 131072
        slaveID = (AbsChannelID - 131072 * portID) // 4096
        chipID = (AbsChannelID - 4096 * slaveID - 131072 * portID) // 64
        channelID = AbsChannelID % 64
        return 3072 * portID + 1024 * slaveID + 64 * chipID + channelID

    # Read coordinate CSV
    mapping_df = pd.read_csv(
        coord_csv_path,
        usecols=[0, 1, 2],
        header=None,
        names=["position x", "position y", "position z"]
    )

    # Validate axis
    coord_axis = coord_axis.lower()
    if coord_axis not in ['x', 'y', 'z']:
        raise ValueError("coord_axis must be one of 'x', 'y', or 'z'.")

    # Precompute sorted coordinate groupings
    full_coords = mapping_df[f"position {coord_axis}"].dropna().unique()
    unique_coords_desc = sorted(full_coords, reverse=True)
    unique_coords_desc = unique_coords_desc[1:]
    coord_groups = [
        unique_coords_desc[i * group_size:(i + 1) * group_size]
        for i in range((len(unique_coords_desc) + group_size - 1) // group_size)
    ]

    grouped_dict = defaultdict(list)  # Temporary storage for grouped chunks
    dfs = pd.DataFrame()
    # print(coord_groups)
    # print(group_size)
    # print(len(coord_groups))

    # Read the PET dataset in chunks
    for chunk in pd.read_csv(os.path.join(in_dir, data_file), delimiter="\t", chunksize=chunksize):
        chunk.columns = ["TimeL", "ChargeL", "ChannelIDL", "TimeR", "ChargeR", "ChannelIDR"]

        # Time normalization
        chunk["TimeL"] /= 1e12
        chunk["TimeR"] /= 1e12

        # Compute reco channel IDs
        chunk["RecoChannelIDL"] = chunk["ChannelIDL"].apply(to_reco_channel_id)
        chunk["RecoChannelIDR"] = chunk["ChannelIDR"].apply(to_reco_channel_id)

        # Map to coordinates
        chunk["_coordL"] = chunk["RecoChannelIDL"].map(mapping_df[f"position {coord_axis}"])
        chunk["_coordR"] = chunk["RecoChannelIDR"].map(mapping_df[f"position {coord_axis}"])
        chunk = chunk.dropna(subset=["_coordL", "_coordR"])

        if chunk.empty:
            continue

        # Assign chunk to appropriate coordinate groups
        if group_size == -1:
            dfs = pd.concat([dfs, chunk], ignore_index=True)
        else:
            for group_coords in coord_groups:
                group_mask = chunk["_coordL"].isin(group_coords) & chunk["_coordR"].isin(group_coords)
                group = chunk[group_mask]
                if not group.empty:
                    group_label = round(np.mean(group_coords), 3)
                    grouped_dict[group_label].append(group)

    # Final assembly of grouped data
    if group_size == -1:
        return [dfs]
    final_grouped_dfs = [pd.concat(grouped_dict[label], ignore_index=True) for label in sorted(grouped_dict)]
    return final_grouped_dfs

In [ ]:
def to_reco_channel_id(AbsChannelID):
    portID = AbsChannelID // 131072
    slaveID = (AbsChannelID - 131072 * portID) // 4096
    chipID = (AbsChannelID - 4096 * slaveID - 131072 * portID) // 64
    channelID = AbsChannelID % 64
    return 3072 * portID + 1024 * slaveID + 64 * chipID + channelID




# Read coordinate CSV
mapping_df = pd.read_csv(
    'C:/Users/burri/Documents/PET/Analysis_Programs/Analysis_Programs/Isotope_Fitting/file_generation/TPPT_Scanner_map.csv',
    usecols=[0, 1, 2],
    header=None,
    names=["position x", "position y", "position z"]
)

#coord_axis = coord_axis.lower()
#if coord_axis not in ['x', 'y', 'z']:
#    raise ValueError("coord_axis must be one of 'x', 'y', or 'z'.")

coord_axis = 'x'
group_size = 5

# Precompute sorted coordinate groupings 
full_coords = mapping_df[f"position {coord_axis}"].dropna().unique()
unique_coords_desc = sorted(full_coords, reverse=True)
unique_coords_desc = unique_coords_desc[1:]
coord_groups = [
    unique_coords_desc[i * group_size:(i + 1) * group_size]
    for i in range((len(unique_coords_desc) + group_size - 1) // group_size)
]

grouped_dict = defaultdict(list)  # Temporary storage for grouped chunks
dfs = pd.DataFrame()


for chunk in pd.read_csv('C:/Users/burri/Documents/PET/photopeak_WIndow_Cut_2024_Flash/photopeak_WIndow_Cut_2024_Flash/FilteredData_Run1_FullAcrylicNo1_Collimator19.75cm_72.5MeV_HWTrigOn_15min_coinc.dat', delimiter="\t", chunksize=100000):
    chunk.columns = ["TimeL", "ChargeL", "ChannelIDL", "TimeR", "ChargeR", "ChannelIDR"]
    
    #Time normalize (into nanoseconds)
    chunk["TimeL"] /= 1e12
    chunk["TimeR"] /= 1e12

    # To reco channel ID
    chunk["RecoChannelIDL"] = chunk["ChannelIDL"].apply(to_reco_channel_id)
    chunk["RecoChannelIDR"] = chunk["ChannelIDR"].apply(to_reco_channel_id)
    
    chunk["_coordLx"] = chunk["RecoChannelIDL"].map(mapping_df[f"position x"])
    chunk["_coordRx"] = chunk["RecoChannelIDR"].map(mapping_df[f"position x"])

    chunk["_coordLy"] = chunk["RecoChannelIDL"].map(mapping_df[f"position y"])
    chunk["_coordRy"] = chunk["RecoChannelIDR"].map(mapping_df[f"position y"])

    dfs = pd.concat([dfs, chunk], ignore_index=True)

print(dfs)





'''
for chunk in pd.read_csv('C:/Users/burri/Documents/PET/photopeak_WIndow_Cut_2024_Flash/photopeak_WIndow_Cut_2024_Flash/FilteredData_Run1_FullAcrylicNo1_Collimator19.75cm_72.5MeV_HWTrigOn_15min_coinc.dat', delimiter="\t", chunksize=100000):
    chunk.columns = ["TimeL", "ChargeL", "ChannelIDL", "TimeR", "ChargeR", "ChannelIDR"]

    # Time normalization
    chunk["TimeL"] /= 1e12
    chunk["TimeR"] /= 1e12

    # Compute reco channel IDs
    chunk["RecoChannelIDL"] = chunk["ChannelIDL"].apply(to_reco_channel_id)
    chunk["RecoChannelIDR"] = chunk["ChannelIDR"].apply(to_reco_channel_id)

    # Map to coordinates
    chunk["_coordL"] = chunk["RecoChannelIDL"].map(mapping_df[f"position {coord_axis}"])
    chunk["_coordR"] = chunk["RecoChannelIDR"].map(mapping_df[f"position {coord_axis}"])
    chunk = chunk.dropna(subset=["_coordL", "_coordR"])

    if chunk.empty:
        continue

    # Assign chunk to appropriate coordinate groups
    if group_size == -1:
        dfs = pd.concat([dfs, chunk], ignore_index=True)
    else:
        for group_coords in coord_groups:
            group_mask = chunk["_coordL"].isin(group_coords) & chunk["_coordR"].isin(group_coords)
            group = chunk[group_mask]
            if not group.empty:
                group_label = round(np.mean(group_coords), 3)
                grouped_dict[group_label].append(group)

# Final assembly of grouped data
final_grouped_dfs = [pd.concat(grouped_dict[label], ignore_index=True) for label in sorted(grouped_dict)]

'''


            TimeL    ChargeL  ChannelIDL       TimeR    ChargeR  ChannelIDR  \
0       79.615320  22.355984    135331.0   79.615320  26.440533      4456.0   
1       79.615826  21.595234    131100.0   79.615826  22.432465      4198.0   
2       79.616170  22.880505    131320.0   79.616170  22.027447       411.0   
3       79.616370  23.237751    131353.0   79.616370  27.991596       311.0   
4       79.617440  22.732101    131640.0   79.617440  25.787125       313.0   
...           ...        ...         ...         ...        ...         ...   
74381  909.730440  21.210506    135442.0  909.730440  25.032250      8309.0   
74382  909.735800  24.242230    131177.0  909.735800  22.059060       425.0   
74383  909.746150  23.008617    132023.0  909.746150  29.125416      8324.0   
74384  909.755200  20.903416    135535.0  909.755200  23.266552      4453.0   
74385  909.761400  22.914474    131212.0  909.761400  19.212006       439.0   

       RecoChannelIDL  RecoChannelIDR    _coordLx  

'\nfor chunk in pd.read_csv(\'C:/Users/burri/Documents/PET/photopeak_WIndow_Cut_2024_Flash/photopeak_WIndow_Cut_2024_Flash/FilteredData_Run1_FullAcrylicNo1_Collimator19.75cm_72.5MeV_HWTrigOn_15min_coinc.dat\', delimiter="\t", chunksize=100000):\n    chunk.columns = ["TimeL", "ChargeL", "ChannelIDL", "TimeR", "ChargeR", "ChannelIDR"]\n\n    # Time normalization\n    chunk["TimeL"] /= 1e12\n    chunk["TimeR"] /= 1e12\n\n    # Compute reco channel IDs\n    chunk["RecoChannelIDL"] = chunk["ChannelIDL"].apply(to_reco_channel_id)\n    chunk["RecoChannelIDR"] = chunk["ChannelIDR"].apply(to_reco_channel_id)\n\n    # Map to coordinates\n    chunk["_coordL"] = chunk["RecoChannelIDL"].map(mapping_df[f"position {coord_axis}"])\n    chunk["_coordR"] = chunk["RecoChannelIDR"].map(mapping_df[f"position {coord_axis}"])\n    chunk = chunk.dropna(subset=["_coordL", "_coordR"])\n\n    if chunk.empty:\n        continue\n\n    # Assign chunk to appropriate coordinate groups\n    if group_size == -1:\n     